In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

anuj232003_nlp_project_dataset_path = kagglehub.dataset_download('anuj232003/nlp-project-dataset')
anuj232003_nlp_checkpoint_path = kagglehub.dataset_download('anuj232003/nlp-checkpoint')

print('Data source import complete.')


In [ ]:
pip install -U transformers peft trl datasets accelerate bitsandbytes sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import gc
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# =========================================================
# Clean memory
# =========================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================================================
# Config
# =========================================================

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DATA_DIR = "/kaggle/input/datasets/anuj232003/nlp-project-dataset"

TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
TEST_FILE = os.path.join(DATA_DIR, "test.jsonl")

OUTPUT_DIR = "mistral_children_story_lora_ckpt"
SEED = 42

# Safer Kaggle settings
MAX_SEQ_LENGTH = 512

# LoRA
LORA_R = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
NUM_EPOCHS = 1
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 3

SYSTEM_PROMPT = """You are a children's story writer.

Write safe, warm, easy-to-understand stories for children ages 6 to 8.
Follow the requested format exactly.
"""

set_seed(SEED)

# =========================================================
# Tokenizer
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# =========================================================
# 4-bit quantization config
# =========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# =========================================================
# Model
# =========================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

# Important for QLoRA
model = prepare_model_for_kbit_training(model)

# =========================================================
# Dataset
# =========================================================

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
        "test": TEST_FILE,
    },
)

print(dataset)
print("\nSample raw row:")
print(dataset["train"][0])

def to_conversational_prompt_completion(example):
    instruction = example["instruction"].strip()
    user_input = example["input"].strip()
    output = example["output"].strip()

    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"{instruction}\n\n{user_input}",
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": output,
            }
        ],
    }

dataset = dataset.map(
    to_conversational_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

print("\nConverted dataset:")
print(dataset)
print("\nSample converted row:")
print(dataset["train"][0])

# =========================================================
# LoRA config
# =========================================================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

# =========================================================
# Training config
# =========================================================

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LENGTH,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=False,
    fp16=False,
    lr_scheduler_type="cosine",
    report_to="none",
    seed=SEED,
    packing=False,
    max_grad_norm=0.5,
    completion_only_loss=True,
    optim="paged_adamw_8bit",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print("\n===== DEBUG SETTINGS =====")
print("bnb compute dtype:", bnb_config.bnb_4bit_compute_dtype)
print("training fp16:", training_args.fp16)
print("training bf16:", training_args.bf16)
print("max_length:", training_args.max_length)
print("learning_rate:", training_args.learning_rate)
print("train batch size:", training_args.per_device_train_batch_size)
print("eval batch size:", training_args.per_device_eval_batch_size)
print("grad_accum_steps:", training_args.gradient_accumulation_steps)
print("eval_steps:", training_args.eval_steps)
print("save_steps:", training_args.save_steps)
print("==========================\n")

# =========================================================
# Trainer
# =========================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
)

# =========================================================
# Train
# =========================================================

print("\nStarting training...")
train_result = trainer.train()

# =========================================================
# Save final model
# =========================================================

print("\nSaving adapter + tokenizer...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save train metrics
train_metrics = train_result.metrics
with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

# Save trainer log history
with open(os.path.join(OUTPUT_DIR, "trainer_log_history.json"), "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

# =========================================================
# Evaluate after training
# =========================================================

print("\nEvaluating best/final model on validation set...")
val_metrics = trainer.evaluate(dataset["validation"])
print(val_metrics)

print("\nEvaluating best/final model on test set...")
test_metrics = trainer.evaluate(dataset["test"])
print(test_metrics)

with open(os.path.join(OUTPUT_DIR, "val_metrics.json"), "w") as f:
    json.dump(val_metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump(test_metrics, f, indent=2)

print(f"\nDone. Outputs saved to: {OUTPUT_DIR}")

In [ ]:
!zip -r submission.zip /kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100


In [ ]:
!zip -r output_files.zip /kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100


In [ ]:
from IPython.display import FileLink
FileLink(r'output_files.zip') # Use the name of your zip file

In [ ]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# =========================================================
# Paths
# =========================================================

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "/kaggle/input/datasets/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100"   # change this if needed

# =========================================================
# Quantization config
# =========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# =========================================================
# Load tokenizer
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================================================
# Load base model
# =========================================================

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# =========================================================
# Attach LoRA adapter
# =========================================================

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# =========================================================
# Prompt template
# =========================================================

SYSTEM_PROMPT = """You are a children's story writer.

Write safe, warm, easy-to-understand stories for children ages 6 to 8.
Follow the requested format exactly.
"""

INSTRUCTION = """Write a children's story for ages 6 to 8.

Requirements:
- Use simple vocabulary and clear sentences
- Keep the story under 500 words
- Use 2 to 3 main characters
- Make the story suitable for 6 to 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, dark, or adult themes

Output format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

def generate_story(theme: str, max_new_tokens: int = 400):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{INSTRUCTION}\n\nTheme: {theme}"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.08,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()

# =========================================================
# Test on a few themes
# =========================================================

themes = ["kindness", "honesty", "sharing", "patience", "teamwork"]

for theme in themes:
    print("\n" + "=" * 80)
    print(f"THEME: {theme}")
    print("=" * 80)
    print(generate_story(theme))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


THEME: kindness
Title: Sammy and Billy
Characters: Sammy, Billy
Story:
Once upon a time in a small town lived two friends named Sammy and Billy. They were best of friends who did everything together. One sunny day, they decided to go on an adventure. They packed their bags filled with sandwiches, water bottles, and toys. Off they went!

As they walked, they saw something unusual. A little bird had fallen from its nest, scared and shivering. It looked sad and helpless. Both friends felt bad seeing the poor bird so distressed. Sammy quickly ran towards the bird, while Billy stayed back to guard their things.

Sammy gently picked up the injured bird, cradling it carefully in his hands. He wrapped it snugly with leaves and placed it in his bag. Together, they continued their journey, making sure not to disturb the little creature.

When they reached a nearby tree, they found a nice spot for their picnic. They ate their sandwiches and shared stories while keeping an eye on their new compan

In [ ]:
theme = "kindness"
output = generate_story(theme)
print(output)

Title: Buddy Bears
Characters: Timmy, Benny
Story:
Once upon a time in Bearville Forest lived two little bear cubs named Timmy and Benny. They were friends who loved playing together, exploring their home, and making new friends. One sunny morning, they decided to take a picnic lunch and go on a hike up Mount Peak. They packed sandwiches, juice boxes, and lots of yummy treats!

As they climbed the mountain trail, they met many animals along the way. They shared some delicious treats and even made some new friends like Tilly the Tortoise and Freddie the Fox. Everyone was so happy, playing games, sharing stories, and learning from one another.

After hours of fun, they reached the top of the mountain where the most beautiful view of Bearville Forest awaited them. Timmy and Benny sat down to enjoy their delicious picnic, watching the sunset with their new friends. They felt grateful for the adventure and excited about all the new memories they had created that day.

Suddenly, nightfall ar